
# 練習問題: モンテカルロ法で円周率 π を推定する (各スレッドが独立に)

## 目標

work-sharing 構文 (`for` / `do`) や `reduction` はまだ学んでいない. ここでは `omp_get_thread_num()` と `omp_get_num_threads()` だけを使って, モンテカルロ法による π の推定を複数スレッドで行う. 各スレッドは**自分の担当分の点を投げ, 自分自身の π 推定値を表示する**. 共有変数への足し込み (集約) はまだ行わない.

## 背景: モンテカルロ法

単位正方形 `[0,1) x [0,1)` 内に一様乱数で点を投げると, 原点中心・半径 1 の 1/4 円 (`x^2 + y^2 < 1`) の内側に落ちる確率は, その面積比 `π/4` に等しい. したがって

```
(円内に落ちた点の数) / (投げた点の総数) ≒ π/4
```

なので, この比を 4 倍すれば π の推定値が得られる. 点を多く投げるほど π ≒ 3.14159 に近づく.

## 課題

`montecarlo.cpp` (または `montecarlo.f90`) は, 全体で `N` 個 (コマンドライン引数, 既定 4,000,000) の点を投げる. 各スレッド `t` (全 `T` スレッド) は `N/T` 個の点を投げ, 円内に落ちた数を数えて自分の π 推定値を表示する.

コメント `TODO` の指示に従って **OpenMP の指示行を追加** し, このブロックを複数スレッドで実行させよ.

- C++: ブロック `{ ... }` の直前に `#pragma omp parallel` を1行加える. スレッドごとの変数 (`tid`, `hits` など) はブロック内で宣言してあるので, 自動的にスレッドごとに別々になる.
- Fortran: ブロックを `!$omp parallel private(...)` と `!$omp end parallel` で囲む. スレッドごとに別々にしたい変数を `private` 節に並べる.

**注意:** 1つの共有変数に全スレッドの `hits` を足し込んではならない (それは競合 (data race) になる. 総和をまとめる `reduction` は後のトピック). 各スレッドは**自分の**推定値だけを表示すること.

乱数は **状態を持たない (カウンタベースの) 純粋関数 `draw_rand01(seed, k)`** を使っている. 点 `i` の座標を `draw_rand01(i, 0)`, `draw_rand01(i, 1)` で決めるので, どのスレッドが計算しても点 `i` の位置は同じで, 共有状態が無いため競合も起きない. これは既に書かれているので変更不要. (同じ仕組みは `06_reduction/02_montecarlo_pi` や `14_montecarlo/00_gacha` でも使う.)

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore montecarlo.cpp -o montecarlo.exe

# Fortran
nvfortran -fast -mp=multicore montecarlo.f90 -o montecarlo.exe
```

```
OMP_NUM_THREADS=4 ./montecarlo.exe
OMP_NUM_THREADS=4 ./montecarlo.exe 40000000
```

## 期待される結果

`OMP_NUM_THREADS` に指定した数だけ行が表示され, 各スレッドが自分の推定値を出す (順番は実行ごとに入れ替わってよい). 各推定値は π ≒ 3.14159 に近い値になる. 例 (4スレッド):

```
thread 0 of 4: 1000000 points, pi estimate = 3.141234
thread 1 of 4: 1000000 points, pi estimate = 3.140876
thread 2 of 4: 1000000 points, pi estimate = 3.142560
thread 3 of 4: 1000000 points, pi estimate = 3.141100
```

投げる点の総数 `N` を増やすほど, 各推定値が π に近づくことを確認せよ. (穴あきのまま実行すると 1 スレッドだけが動き, 行が1つしか出ない.)



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ montecarlo.cpp
#include <cstdio>
#include <cstdlib>
#include <omp.h>

/* 状態を持たない (カウンタベースの) 乱数: (seed,k) から [0,1) の値を決める純粋関数。
   点 i の座標を draw_rand01(i,0), draw_rand01(i,1) で決めるので, どのスレッドが
   担当しても点 i の位置は同じ (共有状態が無いので競合しない)。
   (教育用の簡単なハッシュ。M=2^31-1 未満で計算し, 途中の積も 64bit に収まる。) */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;   /* 2^31 - 1 */
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;        /* [0,1) */
}

int main(int argc, char ** argv) {
  // 全体で投げる点の数 (コマンドライン引数, 既定 4,000,000)
  long N = (argc > 1) ? atol(argv[1]) : 4000000L;
  // TODO: 下のブロックの直前に #pragma omp parallel を1行追加し, 各スレッドが自分の担当分の点を投げて, 自分の π 推定値を表示するようにせよ.
  {
    int tid = omp_get_thread_num();
    int nt  = omp_get_num_threads();
    // このスレッドの担当する点の範囲 (全体 N 点を T スレッドで分割)
    long lo = (long)tid * N / nt;
    long hi = (long)(tid + 1) * N / nt;
    long my_n = hi - lo;
    long hits = 0;
    for (long i = lo; i < hi; i++) {
      double x = draw_rand01(i, 0);   // 点 i の x 座標
      double y = draw_rand01(i, 1);   // 点 i の y 座標
      if (x * x + y * y < 1.0) {
        hits++;
      }
    }
    // 単位正方形に対する 1/4 円の面積比 = π/4
    double pi = (my_n > 0) ? 4.0 * hits / my_n : 0.0;
    printf("thread %d of %d: %ld points, pi estimate = %f\n",
           tid, nt, my_n, pi);
  }
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore montecarlo.cpp -o montecarlo_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./montecarlo_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ montecarlo.f90
program montecarlo
  use omp_lib
  implicit none
  integer(8) :: n, lo, hi, my_n, hits, i
  integer :: tid, nt
  real(8) :: x, y, pi
  character(len=32) :: arg

  ! 全体で投げる点の数 (コマンドライン引数, 既定 4,000,000)
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg)
     read(arg, *) n
  else
     n = 4000000_8
  end if

  ! TODO: 下のブロックを !$omp parallel private(tid, nt, lo, hi, my_n, hits, i, x, y, pi) ... !$omp end parallel で囲み, 各スレッドが自分の担当分の点を投げて自分の π 推定値を表示するようにせよ.
  tid = omp_get_thread_num()
  nt  = omp_get_num_threads()
  ! このスレッドの担当する点の範囲 (全体 n 点を T スレッドで分割)
  lo = tid * n / nt
  hi = (tid + 1) * n / nt
  my_n = hi - lo
  hits = 0
  do i = lo, hi - 1
     x = draw_rand01(i, 0_8)         ! 点 i の x 座標
     y = draw_rand01(i, 1_8)         ! 点 i の y 座標
     if (x * x + y * y < 1.0d0) then
        hits = hits + 1
     end if
  end do
  ! 単位正方形に対する 1/4 円の面積比 = π/4
  if (my_n > 0) then
     pi = 4.0d0 * real(hits, 8) / real(my_n, 8)
  else
     pi = 0.0d0
  end if
  print "(a,i0,a,i0,a,i0,a,f0.6)", &
       "thread ", tid, " of ", nt, ": ", my_n, " points, pi estimate = ", pi
  ! TODO: 上で始めた parallel 領域を閉じる (!$omp end parallel).

contains

  ! 状態を持たない (カウンタベースの) 乱数: (seed,k) から [0,1) の値を決める純粋関数。
  ! 点 i の座標を draw_rand01(i,0), draw_rand01(i,1) で決めるので, どのスレッドが
  ! 担当しても点 i の位置は同じ (共有状態が無いので競合しない)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8   ! 2^31 - 1
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)        ! [0,1)
  end function draw_rand01

end program montecarlo

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore montecarlo.f90 -o montecarlo_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./montecarlo_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:montecarlo.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:montecarlo.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:montecarlo.cpp}